# Solutions: Replicating Varian (2014), growth regressions

**Course:** EAIF: AI for Finance. **Activity:** read before class, replicate in class, reflect.

**Paper:** Hal R. Varian (2014), "Big Data: New Tricks for Econometrics", *Journal of Economic Perspectives* 28(2): 3-28.
- Article: https://www.aeaweb.org/articles?id=10.1257/jep.28.2.3
- Replication data (provenance, no download needed): OpenICPSR project 113925, https://www.openicpsr.org/openicpsr/project/113925

## What we replicate

Section 6.3 of the paper ("Economic example: growth regressions") compares four **variable-selection** methods on one small dataset: Bayesian model averaging (BMA, Ley and Steel 2009), Sala-i-Martin's CDF(0), the **lasso**, and spike-and-slab regression. The result is one table (Table 4 in the working-paper version) listing the ten predictors of economic growth that BMA ranks highest, with each method's verdict next to them. Varian's conclusion is that the four methods agree on the first four or five variables and then diverge, and that the dataset is too small to resolve what is "important" for growth.

The data are in `FLS-data.csv` (72 countries, 41 regressors, one outcome). We fit the lasso column of that table ourselves, then ask what a linear regression and a random forest say about the same data.

**Workflow**

1. Load the data (one line, from the course repository).
2. Inspect it and read the codebook.
3. Split into training and test rows.
4. Lasso: which variables survive, and in which order do they enter? Compare with the BMA ranking.
5. OLS baseline: what happens with 42 parameters and about 54 rows?
6. Random forest: does a non-linear learner rank the same variables?
7. Compare, then answer the reflection questions.

> You may use AI tools for coding help, but make sure you understand the code.


## 0) Setup
Run this cell first. Everything used below is preinstalled in Google Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LassoCV, lasso_path
from sklearn.ensemble import RandomForestRegressor

# root_mean_squared_error exists from scikit-learn 1.4 on; older versions need the fallback
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    def root_mean_squared_error(y_true, y_pred):
        return np.sqrt(mean_squared_error(y_true, y_pred))

plt.rcParams['figure.dpi'] = 110
print('Ready. pandas', pd.__version__)


## 1) Get the data

**Option A (default):** the file is hosted in the course repository and loads with one line.

**Option B (fallback, only if Option A fails):** download `FLS-data.csv` from the OpenICPSR replication package (folder "Lasso") and upload it when the dialog appears.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/umatter/EDFB/main/activity/FLS-data.csv"

try:
    df = pd.read_csv(DATA_URL)                       # Option A
    print(f"Loaded from the course repository: {df.shape[0]} rows x {df.shape[1]} columns")
except Exception as e:                               # Option B: manual upload in Colab
    print("Could not load from the repository:", e)
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded. Run the cell again and choose FLS-data.csv.")
    csv_name = list(uploaded.keys())[0]
    df = pd.read_csv(csv_name)
    print(f"Loaded uploaded file {csv_name}: {df.shape[0]} rows x {df.shape[1]} columns")


### Codebook

`FLS-data.csv` is the cross-country growth dataset of Fernandez, Ley and Steel (2001), "Model uncertainty in cross-country growth regressions", *Journal of Applied Econometrics* 16: 563-576. It goes back to Sala-i-Martin (1997) and was reused by Ley and Steel (2009) and by Varian (2014), Section 6.3.

- **Rows:** 72 countries.
- **`y`:** average growth rate of GDP per capita, 1960 to 1992 (percent per year).
- **41 regressors**, for example `GDPsh560` (log GDP per capita in 1960), `Confuncious` (fraction Confucian; the spelling is the original file's), `Life Exp` (life expectancy), `Equip Inv` (equipment investment share), `SubSahara` (dummy), `Muslim`, `Rule of Law`, `Yrs Open` (fraction of years 1950-1994 the economy was open), `Eco Org` (degree of capitalism), `Protestants`, and so on.
- **Column order matters:** the regressors appear in the order of the BMA posterior inclusion probability reported in Varian's table (Ley and Steel's ranking). The first ten columns after `y` are the ten rows of that table. We use this order as the "BMA ranking" below.


## 2) Inspect
Look at the structure, then set the target and the features. The target is `y`; every other column is a candidate regressor.

In [ ]:
print(df.shape)
display(df.head())
display(df.describe().T.round(3))
print("Missing values in total:", int(df.isna().sum().sum()))

target_col = 'y'
feature_cols = [c for c in df.columns if c != target_col]
bma_rank = pd.Series(range(1, len(feature_cols) + 1), index=feature_cols, name='BMA_rank')
print(f"\nTarget: {target_col}; {len(feature_cols)} regressors. First ten by BMA ranking:")
print(list(bma_rank.index[:10]))


## 3) Train/test split
We hold out a quarter of the countries (18 of 72) to measure out-of-sample error. With so few rows, the test error of any model is noisy: a different `random_state` can change the ranking of models. Keep that in mind when you read the numbers.

In [ ]:
X = df[feature_cols].copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Training rows: {len(X_train)}, test rows: {len(X_test)}, regressors: {X.shape[1]}")

# Baseline every model must beat: predict the training mean
baseline_rmse = root_mean_squared_error(y_test, np.full(len(y_test), y_train.mean()))
print(f"Naive baseline (training mean) test RMSE: {baseline_rmse:.3f}")


## 4) Lasso: replicate the variable selection

Two things are computed here.

1. **Which variables survive.** `LassoCV` picks the penalty by 5-fold cross-validation on the training rows and returns the coefficients at that penalty. Regressors are standardised first, because the lasso penalises all coefficients equally and the columns are on very different scales.
2. **In which order variables enter the path.** As the penalty is relaxed, variables get non-zero coefficients one after another. That entry order is what Varian's "lasso" column reports (1 = first to enter). We compute the path on all 72 countries, as the paper did, and put the entry order next to the BMA ranking encoded in the column order.


In [ ]:
# (1) LassoCV on the training rows, penalty chosen by 5-fold CV
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

lasso = LassoCV(cv=5, random_state=42, max_iter=20000).fit(X_train_s, y_train)
lasso_coef = pd.Series(lasso.coef_, index=feature_cols)
survivors = lasso_coef[lasso_coef != 0].sort_values(key=abs, ascending=False)
print(f"Penalty chosen by CV: alpha = {lasso.alpha_:.4f}")
print(f"Variables with non-zero coefficient: {len(survivors)} of {len(feature_cols)}")
print(survivors.round(3))

y_pred_lasso = lasso.predict(X_test_s)
lasso_rmse = root_mean_squared_error(y_test, y_pred_lasso)
lasso_mae = mean_absolute_error(y_test, y_pred_lasso)
lasso_r2 = r2_score(y_test, y_pred_lasso)
print(f"\nLasso test RMSE {lasso_rmse:.3f} (naive baseline {baseline_rmse:.3f}), MAE {lasso_mae:.3f}, R2 {lasso_r2:.3f}")


In [ ]:
# (2) Entry order along the lasso path, on all 72 countries (standardised)
X_all_s = StandardScaler().fit_transform(X)
# Explicit penalty grid from the largest penalty that zeroes every coefficient down to 1/1000 of it
alpha_max = np.max(np.abs(X_all_s.T @ (y.values - y.values.mean()))) / len(y)
alpha_grid = np.logspace(np.log10(alpha_max), np.log10(alpha_max * 1e-3), 300)
alphas, coefs, _ = lasso_path(X_all_s, y.values, alphas=alpha_grid)

# For each variable: the first grid point (largest penalty) at which its coefficient is non-zero
first_nonzero = [(np.flatnonzero(np.abs(coefs[j]) > 0)[0] if np.any(coefs[j] != 0) else np.inf)
                 for j in range(coefs.shape[0])]
entry = pd.DataFrame({'first_step': first_nonzero}, index=feature_cols)
entry['lasso_entry_order'] = entry['first_step'].rank(method='min').astype(int)
entry['BMA_rank'] = bma_rank

print("Varian's table rows (BMA ranking) with the lasso entry order from our path:")
print(entry.loc[bma_rank.index[:10], ['BMA_rank', 'lasso_entry_order']])
print("\nFirst ten variables to enter our lasso path:")
print(entry.sort_values('lasso_entry_order').head(10)[['lasso_entry_order', 'BMA_rank']])


**What we see.** In our run the first variables to enter the lasso path were `Equip Inv`, `Yrs Open`, `Confuncious`, `Buddha` and `SubSahara`; `Protestants` and `Eco Org` followed within the first ten. That matches Varian's lasso column closely: his table has equipment investment first, the Confucian fraction second, then Protestant, open economy, Sub-Saharan, Muslim and degree of capitalism. It also reproduces his dashes: `GDPsh560`, `Life Exp` and `Rule of Law`, which BMA ranks first, third and seventh, entered our path only at positions 16, 19 and 17, so at any penalty that keeps the model small they are left out. The exact positions depend on the penalty grid and on standardisation, and yours may shift by a place or two. The pattern is Varian's point: the methods agree on the first four or five variables and then diverge, and 72 countries are too few to say more.

## 5) Baseline: OLS with all 41 regressors

`statsmodels` OLS estimates 42 parameters (41 slopes plus an intercept) from the training rows. With about 54 rows that leaves very few degrees of freedom, and the fit is close to perfect in-sample by construction. The interesting part is the test error.

To compare effect sizes, never sort raw coefficients: `Pop g` is a fraction and `Area` is in millions of square kilometres, so their coefficients live on incomparable scales. We rank by **t-statistic** (coefficient divided by its standard error), which is unit-free, and also show standardised coefficients (effect of a one-standard-deviation change).

In [ ]:
X_train_c = sm.add_constant(X_train)
X_test_c = sm.add_constant(X_test)
ols_model = sm.OLS(y_train, X_train_c).fit()

y_pred_ols = ols_model.predict(X_test_c)
ols_rmse = root_mean_squared_error(y_test, y_pred_ols)
ols_mae = mean_absolute_error(y_test, y_pred_ols)
ols_r2 = r2_score(y_test, y_pred_ols)
print(f"OLS: {int(ols_model.df_model) + 1} parameters, {int(ols_model.df_resid)} residual degrees of freedom")
print(f"In-sample R2 {ols_model.rsquared:.3f}; test RMSE {ols_rmse:.3f} (naive baseline {baseline_rmse:.3f}), MAE {ols_mae:.3f}, R2 {ols_r2:.3f}")

# Rank effects by t-statistic, and show standardised coefficients (beta * sd(x))
ols_table = pd.DataFrame({
    'coef': ols_model.params.drop('const'),
    'std_coef': ols_model.params.drop('const') * X_train.std(),
    't': ols_model.tvalues.drop('const'),
    'p': ols_model.pvalues.drop('const'),
})
ols_table['BMA_rank'] = bma_rank
ols_table = ols_table.reindex(ols_table['t'].abs().sort_values(ascending=False).index)
print("\nTop 10 regressors by |t|:")
print(ols_table.head(10).round(3))


## 6) Extension: random forest

Varian does not fit a random forest to these data. We add one to ask what a non-linear learner makes of 72 rows. Tune a small grid by 5-fold CV on the training rows; the test rows are used once.

In [ ]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
param_grid = {'n_estimators': [300], 'max_depth': [None, 4, 8], 'min_samples_leaf': [1, 3, 5]}
gs = GridSearchCV(rf, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
gs.fit(X_train, y_train)
print('Best parameters:', gs.best_params_)
print(f"CV RMSE at best parameters: {-gs.best_score_:.3f}")

best_rf = gs.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_rmse = root_mean_squared_error(y_test, y_pred_rf)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)
print(f"Random forest test RMSE {rf_rmse:.3f} (naive baseline {baseline_rmse:.3f}), MAE {rf_mae:.3f}, R2 {rf_r2:.3f}")

rf_importance = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
rf_top = pd.DataFrame({'importance': rf_importance.head(10), 'BMA_rank': bma_rank[rf_importance.head(10).index]})
print("\nTop 10 by random-forest importance:")
print(rf_top.round(3))
rf_importance.head(10)[::-1].plot(kind='barh', title='Random forest: top 10 feature importances')
plt.xlabel('Importance'); plt.tight_layout(); plt.show()


## 7) Compare
One table of test errors next to the naive baseline, one plot of predictions, and the three variable rankings side by side.

In [ ]:
results = pd.DataFrame({
    'Naive mean': [baseline_rmse, mean_absolute_error(y_test, np.full(len(y_test), y_train.mean())), 0.0],
    'Lasso': [lasso_rmse, lasso_mae, lasso_r2],
    'OLS': [ols_rmse, ols_mae, ols_r2],
    'Random forest': [rf_rmse, rf_mae, rf_r2],
}, index=['RMSE', 'MAE', 'R2'])
print("Test-set performance (18 countries):")
print(results.round(3))

# Cross-validated RMSE on the training rows, for a second opinion that does not depend on one split
# Scaling inside each fold, so that the validation rows do not touch the scaler
cv_lasso = -cross_val_score(make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=42, max_iter=20000)),
                            X_train, y_train, cv=5, scoring='neg_root_mean_squared_error').mean()
cv_ols = -cross_val_score(LinearRegression(), X_train, y_train, cv=5, scoring='neg_root_mean_squared_error').mean()
cv_rf = -gs.best_score_
print(f"\n5-fold CV RMSE on training rows: lasso {cv_lasso:.3f}, OLS {cv_ols:.3f}, random forest {cv_rf:.3f}")

plt.figure()
plt.scatter(y_test, y_pred_lasso, alpha=0.7, label='Lasso')
plt.scatter(y_test, y_pred_ols, alpha=0.7, label='OLS')
plt.scatter(y_test, y_pred_rf, alpha=0.7, label='Random forest')
lims = [float(y_test.min()), float(y_test.max())]
plt.plot(lims, lims, linestyle='--', color='grey')
plt.xlabel('Actual growth (% per year)'); plt.ylabel('Predicted'); plt.legend()
plt.title('Test countries: actual vs predicted'); plt.tight_layout(); plt.show()

rankings = pd.DataFrame({
    'BMA (column order)': bma_rank.index[:10],
    'Lasso entry order': entry.sort_values('lasso_entry_order').index[:10],
    'OLS |t|': ols_table.index[:10],
    'RF importance': rf_importance.index[:10],
}, index=range(1, 11))
print("\nTop ten variables under each method:")
print(rankings.to_string())


## 8) What our run found

The numbers below are from our run (`random_state=42`, `test_size=0.25`, 54 training and 18 test countries). Yours will differ a little; the pattern should not.

**Test-set performance (18 countries), in our run**

| | Naive mean | Lasso | OLS | Random forest |
|---|---|---|---|---|
| RMSE | 1.75 | 1.01 | 1.26 | 1.14 |
| R2 | 0.00 | 0.65 | 0.46 | 0.56 |
| 5-fold CV RMSE (training rows, scaling inside the folds) | | 1.63 | 4.89 | 1.31 |

Three things to notice.

1. **All three models beat the naive mean on the test countries**, but the ranking is close (1.01 vs 1.14 vs 1.26 on 18 observations) and would not survive a different split. Do not present it as "the lasso wins".
2. **OLS is the one that overfits.** In-sample R2 was 0.98 with 12 residual degrees of freedom, and the cross-validated RMSE (4.89) is far worse than its test RMSE (1.26): in some folds the 42-parameter fit extrapolates wildly. This is the situation Section 3 of the paper describes: n regressors fit n observations perfectly and predict badly.
3. **The lasso and the forest are regularised**: their CV errors (1.63 and 1.31) stay in the range of their test errors instead of exploding as OLS does. The lasso kept 17 of 41 regressors at the CV-chosen penalty in our run.

### A1. What OLS found (ranked by |t|, not by coefficient size)

In our run the largest t-statistics belonged to `GDPsh560` (t about -4.6: richer countries in 1960 grew more slowly, the convergence result), `Mining` (+3.5), `High Enroll` (-3.4, an unexpected sign), `Spanish Col` (+2.4) and `Hindu` (-2.1). Only three or four coefficients are significant at 5%; the rest are noise with wide standard errors. Sorting by raw coefficient would have put `High Enroll` (-18.0) and `Hindu` (-10.7) at the top and hidden `GDPsh560` (-2.2), because those variables are fractions and initial GDP is a log level. Standardised coefficients tell the same story as the t-statistics: a one-standard-deviation increase in initial GDP is worth about -1.8 percentage points of growth, the largest standardised effect in the table.

### A2. What the random forest found

In our run the top importances were `Equip Inv` (0.24), `NEquip Inv` (0.11), `Buddha` (0.11), `Life Exp` (0.07) and `Yrs Open` (0.07). Investment, health and openness dominate; religion shares appear as proxies for East Asian growth. `GDPsh560`, the strongest OLS variable, only reaches tenth place: a forest with 54 rows cannot represent a smooth monotone effect as efficiently as a linear term can.

### A3. Where the methods agree and where they do not

- **Agreement:** `Equip Inv` is first in Varian's lasso column, first in our lasso path and first in the forest; `Yrs Open`, `Confuncious`/`Buddha`, `SubSahara` and `Protestants` appear near the top of the BMA ranking and of our lasso path. `PrSc Enroll` shows up in the OLS and forest top tens.
- **Disagreement:** `GDPsh560` and `Life Exp` sit at the top of the BMA ranking and of the OLS t-statistics but enter the lasso path late and rank low in the forest. `Mining`, `High Enroll`, `Spanish Col` and `Hindu` are OLS-only. `R FEX Dist` is lasso-only; `std(BMP)` appears in the lasso and forest lists but not in BMA or OLS.
- **Why they disagree:** OLS with 12 residual degrees of freedom produces unstable coefficients, so a variable's rank is partly luck of the sample; the lasso trades a little bias for much lower variance and drops variables that are correlated with ones already in (initial GDP is correlated with life expectancy, schooling and the regional dummies); the forest rewards variables that split the sample well, not variables with a linear effect. The four columns of Varian's table disagree for the same reasons. Agreement on the first handful of variables is the robust finding; everything after that is model uncertainty.


## 9) Reflection answers

**1. The growth-regression table (Section 6.3).** Every method we ran, and every column of Varian's table, puts equipment investment at or near the top, and most of them agree on a small East Asian / Confucian cluster, openness and the Sub-Saharan dummy. After that the lists diverge: BMA and OLS like initial GDP and life expectancy, the lasso and the forest do not. Varian's sentence that the dataset "appears to be too small to resolve" what is important for growth is exactly what our table shows: with 72 countries and 41 candidate regressors, several different models fit about equally well and rank the remaining variables differently. There is no single "true" list to be read off.

**2. Prediction versus causation (Sections 3 and 8.1).** Everything we did answers a prediction question: given a country's 1960 characteristics, how fast did it grow? The train/test split and cross-validation of Section 3 measure how well that works out of sample. Section 8.1 asks the other question, whether changing a variable would change the outcome, and illustrates it with police and crime: precincts with more police have more crime, which makes police counts an excellent predictor of crime and says nothing about what sending more police would do, because police were assigned where crime was high. Our growth regressors are the same kind of variable. Equipment investment predicts growth, but investment is high where growth is expected, so the coefficient cannot be read as the return to a policy of subsidising equipment. A policymaker could use the model to forecast, not to decide.

**3. Model uncertainty (Section 9).** Varian argues that applied economists worry about sampling uncertainty (standard errors) and ignore model uncertainty (which specification), even though the second is often larger, and that averaging over many models tends to predict better than picking one. Our results show it twice: the top-ten lists change with the method, and the test RMSE changes with the model (and would change again with the split). Averaging here would mean averaging the predictions of the lasso, the OLS and the forest, or, closer to the paper, averaging over many small linear models weighted by how well they fit, which is what BMA does in the first column of Varian's table.


## 10) Where overfitting and leakage could enter

**Overfitting.** OLS with 42 parameters on 54 rows is the textbook case (Section 3 of the paper). The lasso controls it with a penalty chosen by cross-validation, the forest with tree depth, leaf size and bootstrap averaging. Tuning by cross-validation is itself a fit to the training rows: the grid search picked `max_depth=8` and `min_samples_leaf=1` in our run, and a different grid could pick differently. The test rows are used once at the end for that reason.

**Leakage, three kinds.**

1. *Target leakage:* a regressor that is a consequence of the outcome. Average growth 1960 to 1992 is the target; `GDPsh560` is measured in 1960 and is fine, but a 1992 income level would be the target in disguise. Ask of every column: was it known before the outcome was realised?
2. *Train/test contamination:* fitting the scaler, the lasso penalty or the forest grid on all 72 rows and then evaluating on 18 of them. In the notebook the scaler and `LassoCV` are fitted on the training rows only; the lasso path in Section 4(2) uses all rows deliberately, because it replicates a descriptive table and is not evaluated on the test rows.
3. *Temporal leakage:* not an issue in one cross-section, but it becomes the main issue as soon as the same exercise is run on a panel of years (see the group assignment): train on early years, test on later ones.

**Red flag:** a test error that looks too good, or a variable with an implausibly large importance, is a reason to check the data before celebrating.


## 11) Optional further reading

- Fernandez, Ley and Steel (2001), *Journal of Applied Econometrics* 16: 563-576: the source of the data and of the model-averaging approach.
- Ley and Steel (2009), *Journal of Applied Econometrics* 24: 651-674: the BMA ranking Varian's table is based on.
- Sala-i-Martin (1997), "I just ran two million regressions", *American Economic Review* 87(2): 178-183: the CDF(0) column.
- James, Witten, Hastie and Tibshirani, *An Introduction to Statistical Learning*, chapter 6 (lasso) and chapter 8 (trees and forests): the methods at the level of this course.
